In [1]:
import pandas as pd

In [2]:
files = [
    "../data/raw/ratings.csv",
    "../data/raw/movies.csv",
    "../data/raw/tags.csv",
    "../data/raw/links.csv",
    "../data/raw/genome-tags.csv",
    "../data/raw/genome-scores.csv",
]

In [3]:
df = [pd.read_csv(file) for file in files]

## **Shape of each file**

In [4]:
for index in range(len(df)):
    print(f"{files[index][12:]} : {df[index].shape[0]} rows, {df[index].shape[1]} columns\n")

ratings.csv : 25000095 rows, 4 columns

movies.csv : 62423 rows, 3 columns

tags.csv : 1093360 rows, 4 columns

links.csv : 62423 rows, 3 columns

genome-tags.csv : 1128 rows, 2 columns

genome-scores.csv : 15584448 rows, 3 columns



## **Head of each file**

In [6]:
for name, data in zip(files, df, strict=False):
    print(f"--- Fichier : {name[12:]} ---")
    display(data.head())
    print("\n")

--- Fichier : ratings.csv ---


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510




--- Fichier : movies.csv ---


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy




--- Fichier : tags.csv ---


...

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0




--- Fichier : genome-tags.csv ---


,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s




--- Fichier : genome-scores.csv ---


,movieId,tagId,relevance
0,1,1,0.02875
1,1,2,0.02375
2,1,3,0.06250
3,1,4,0.07575
4,1,5,0.14075


## **Dtypes of each variable**

In [ ]:
for file in df:
    print(file.dtypes)

userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object
movieId    int64
title        str
genres       str
dtype: object
userId       int64
movieId      int64
tag            str
timestamp    int64
dtype: object
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object
tagId    int64
tag        str
dtype: object
movieId        int64
tagId          int64
relevance    float64
dtype: object


## **Missing values**

In [ ]:
for file in df:
    missing_values = file.isnull().sum()
    print(missing_values)

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
movieId    0
title      0
genres     0
dtype: int64
userId        0
movieId       0
tag          16
timestamp     0
dtype: int64
movieId      0
imdbId       0
tmdbId     107
dtype: int64
tagId    0
tag      0
dtype: int64
movieId      0
tagId        0
relevance    0
dtype: int64


### **Observations on missing values:**

* **Intact files:** The main tables (`ratings.csv`, `movies.csv`, and "genome" data) contain **no missing values**. This is ideal for building our baseline recommendation system.
* **Tags file (16 missing values):** 16 keywords are missing in the `tag` column. This means some users likely submitted an empty field.
    * *Planned action:* As these 16 rows are unusable in their current state, we can simply remove them during the cleaning phase.
* **Links file (107 missing values):** 107 `tmdbId` identifiers are missing.
    * *Impact:* This has no impact on the recommendation algorithm itself. It will only pose a problem if we later decide to connect to the TMDB API to retrieve posters or summaries for these specific 107 movies.

## **Minimal and Maximal dates**

In [ ]:
for index in range(len(df)):
    if "timestamp" in df[index].columns:
        min_ts = df[index]["timestamp"].min()
        max_ts = df[index]["timestamp"].max()

        date_min = pd.to_datetime(min_ts, unit="s")
        date_max = pd.to_datetime(max_ts, unit="s")

        print(f"Minimal date : {date_min}")
        print(f"Maximal date : {date_max}")

Minimal date : 1995-01-09 11:46:49
Maximal date : 2019-11-21 09:15:03
Minimal date : 2005-12-24 13:00:10
Maximal date : 2019-11-21 06:11:36


**Observation:** The minimum dates differ between ratings and tags. This indicates that the "tagging" feature was likely introduced to the platform several years after the rating system.

In [ ]:
duplicated_values = df[0].duplicated(subset=["userId", "movieId", "timestamp"]).sum()
print(f"Number of duplicated values in ratings.csv : {duplicated_values}")

duplicated_vote = df[0].duplicated(subset=["userId", "movieId"]).sum()
print(f"Number of duplicated votes in ratings.csv : {duplicated_vote}")

Number of duplicated values in ratings.csv : 0
Number of duplicated votes in ratings.csv : 0


**Observation:** There are no duplicate values in the `ratings.csv` file. No user has rated the same movie twice.